In [1]:
import pandas as pd
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns
import time

In [2]:
def read_mnist_images(path):
    """Read MNIST image binary file"""
    with open(path, 'rb') as f:
        magic = np.frombuffer(f.read(4), dtype='>i4')[0]
        n = np.frombuffer(f.read(4), dtype='>i4')[0]
        rows = np.frombuffer(f.read(4), dtype='>i4')[0]
        cols = np.frombuffer(f.read(4), dtype='>i4')[0]
        images = np.frombuffer(f.read(), np.uint8).reshape(n, rows, cols)
    return images

def read_mnist_labels(path):
    """Read MNIST label binary file"""
    with open(path, 'rb') as f:
        magic = np.frombuffer(f.read(4), dtype='>i4')[0]
        n = np.frombuffer(f.read(4), dtype='>i4')[0]
        labels = np.frombuffer(f.read(), np.uint8)
    return labels

In [3]:
# Load training data
train_images = read_mnist_images('./Dataset/MNIST/train-images.idx3-ubyte')
train_labels = read_mnist_labels('./Dataset/MNIST/train-labels.idx1-ubyte')

# Load test data
test_images = read_mnist_images('./Dataset/MNIST/t10k-images.idx3-ubyte')
test_labels = read_mnist_labels('./Dataset/MNIST/t10k-labels.idx1-ubyte')

print(f"Train images shape: {train_images.shape}")  # (60000, 28, 28)
print(f"Train labels shape: {train_labels.shape}")  # (60000,)
print(f"Test images shape: {test_images.shape}")    # (10000, 28, 28)
print(f"Test labels shape: {test_labels.shape}")  

Train images shape: (60000, 28, 28)
Train labels shape: (60000,)
Test images shape: (10000, 28, 28)
Test labels shape: (10000,)


In [4]:
train_images = train_images / 255.0
test_images = test_images / 255.0

In [5]:
train_images = train_images.reshape(-1, 28*28)
test_images = test_images.reshape(-1, 28*28)

In [6]:
X = train_images
y = train_labels

In [8]:
# Reshape data to 2D (flatten 28x28 images to 784 features)
X_reshaped = X.reshape(X.shape[0], -1)
test_images_reshaped = test_images.reshape(test_images.shape[0], -1)

In [9]:
# ── Train Small MLP with architecture (784, 256, 128, 10) ──────
print("\n=== Training Small MLP (784 -> 256 -> 128 -> 10) ===")
small_model_start = time.time()

small_model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    max_iter=20,
    random_state=42,
    batch_size=128,
    early_stopping=False,
    verbose=1
)
small_model.fit(X_reshaped, y)

small_model_end = time.time()
small_training_time = small_model_end - small_model_start

# Evaluate the small model
y_pred_small = small_model.predict(test_images_reshaped)
small_accuracy = accuracy_score(test_labels, y_pred_small)
small_precision = precision_score(test_labels, y_pred_small, average='weighted')
small_recall = recall_score(test_labels, y_pred_small, average='weighted')
small_f1 = f1_score(test_labels, y_pred_small, average='weighted')

print("\n" + "=" * 40)
print("    SMALL MLP PERFORMANCE")
print("=" * 40)
print(f"Accuracy:   {small_accuracy:.4f}")
print(f"Precision:  {small_precision:.4f}")
print(f"Recall:     {small_recall:.4f}")
print(f"F1-Score:   {small_f1:.4f}")
print(f"Training time: {small_training_time:.4f} s")
print("=" * 40)


=== Training Small MLP (784 -> 256 -> 128 -> 10) ===


/Users/arminsiavashi/Desktop/DCA/CPU-Vs-FPGA/venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/arminsiavashi/Desktop/DCA/CPU-Vs-FPGA/venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/arminsiavashi/Desktop/DCA/CPU-Vs-FPGA/venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Iteration 1, loss = 0.26517303
Iteration 2, loss = 0.10275992
Iteration 3, loss = 0.06937327
Iteration 4, loss = 0.04874538
Iteration 5, loss = 0.03529905
Iteration 6, loss = 0.02868649
Iteration 7, loss = 0.02426809
Iteration 8, loss = 0.01634044
Iteration 9, loss = 0.01494981
Iteration 10, loss = 0.01563122
Iteration 11, loss = 0.01136371
Iteration 12, loss = 0.01109255
Iteration 13, loss = 0.00784656
Iteration 14, loss = 0.01149202
Iteration 15, loss = 0.01116406
Iteration 16, loss = 0.01135700
Iteration 17, loss = 0.00803225
Iteration 18, loss = 0.00824232
Iteration 19, loss = 0.00491734
Iteration 20, loss = 0.00820369

    SMALL MLP PERFORMANCE
Accuracy:   0.9799
Precision:  0.9800
Recall:     0.9799
F1-Score:   0.9799
Training time: 15.3995 s


/Users/arminsiavashi/Desktop/DCA/CPU-Vs-FPGA/venv/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/arminsiavashi/Desktop/DCA/CPU-Vs-FPGA/venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/arminsiavashi/Desktop/DCA/CPU-Vs-FPGA/venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/arminsiavashi/Desktop/DCA/CPU-Vs-FPGA/venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [10]:
# ── Extract Weights and Quantize to 8-bit ────────────────────
import os
from pathlib import Path

# Create output directory for quantized weights
output_dir = Path('./quantized_weights_8bit')
output_dir.mkdir(exist_ok=True)

print("\n=== 8-bit Quantization and Export ===\n")

# Extract weights and biases
coefs = small_model.coefs_  # List of weight matrices
intercepts = small_model.intercepts_  # List of bias vectors

quantized_data = {}
quantization_info = {}

# Quantize each weight matrix and bias
for i, (weight, bias) in enumerate(zip(coefs, intercepts)):
    layer_name = f"dense_{i}"
    
    # 1. Quantize weights
    w_min = weight.min()
    w_max = weight.max()
    
    # Map to [0, 255] range
    w_quantized = ((weight - w_min) / (w_max - w_min) * 255).astype(np.uint8)
    
    # 2. Quantize biases
    b_min = bias.min()
    b_max = bias.max()
    
    b_quantized = ((bias - b_min) / (b_max - b_min) * 255).astype(np.uint8)
    
    # Store quantized values
    quantized_data[f'{layer_name}_weights'] = w_quantized
    quantized_data[f'{layer_name}_bias'] = b_quantized
    
    # Store quantization parameters for dequantization
    quantization_info[f'{layer_name}_weights'] = {
        'min': float(w_min),
        'max': float(w_max),
        'shape': weight.shape
    }
    quantization_info[f'{layer_name}_bias'] = {
        'min': float(b_min),
        'max': float(b_max),
        'shape': bias.shape
    }
    
    print(f"Layer {i} ({layer_name}):")
    print(f"  Weights: shape {weight.shape}, original range [{w_min:.4f}, {w_max:.4f}]")
    print(f"  Bias:    shape {bias.shape}, original range [{b_min:.4f}, {b_max:.4f}]")
    print()

# Save quantized weights as NPY
print("Saving quantized weights as NPY files...")
for name, data in quantized_data.items():
    filepath = output_dir / f'{name}.npy'
    np.save(filepath, data)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  ✓ {name}.npy ({data.shape}) - {size_kb:.2f} KB")

# Save quantization info
import json
info_filepath = output_dir / 'quantization_info.json'
with open(info_filepath, 'w') as f:
    json.dump(quantization_info, f, indent=2)
print(f"  ✓ quantization_info.json")

print(f"\nAll files saved to: {output_dir.absolute()}")


=== 8-bit Quantization and Export ===

Layer 0 (dense_0):
  Weights: shape (784, 256), original range [-0.7715, 0.5032]
  Bias:    shape (256,), original range [-0.2232, 0.1696]

Layer 1 (dense_1):
  Weights: shape (256, 128), original range [-0.6247, 0.6485]
  Bias:    shape (128,), original range [-0.1908, 0.2328]

Layer 2 (dense_2):
  Weights: shape (128, 10), original range [-0.6221, 0.4548]
  Bias:    shape (10,), original range [-0.1260, 0.1520]

Saving quantized weights as NPY files...
  ✓ dense_0_weights.npy ((784, 256)) - 196.12 KB
  ✓ dense_0_bias.npy ((256,)) - 0.38 KB
  ✓ dense_1_weights.npy ((256, 128)) - 32.12 KB
  ✓ dense_1_bias.npy ((128,)) - 0.25 KB
  ✓ dense_2_weights.npy ((128, 10)) - 1.38 KB
  ✓ dense_2_bias.npy ((10,)) - 0.13 KB
  ✓ quantization_info.json

All files saved to: /Users/arminsiavashi/Desktop/DCA/CPU-Vs-FPGA/quantized_weights_8bit


In [11]:
# ── Verify Quantization: Load and Dequantize ──────────────────
print("\n=== Verification: Dequantization ===\n")

# Load quantization info
with open(output_dir / 'quantization_info.json', 'r') as f:
    loaded_quant_info = json.load(f)

# Function to dequantize
def dequantize(quantized_data, quant_info):
    w_min = quant_info['min']
    w_max = quant_info['max']
    return (quantized_data.astype(np.float32) / 255.0) * (w_max - w_min) + w_min

# Load and dequantize weights
reconstructed_weights = []
for i in range(len(coefs)):
    layer_name = f"dense_{i}"
    
    # Load quantized data
    w_quantized = np.load(output_dir / f'{layer_name}_weights.npy')
    b_quantized = np.load(output_dir / f'{layer_name}_bias.npy')
    
    # Dequantize
    w_dequant = dequantize(w_quantized, loaded_quant_info[f'{layer_name}_weights'])
    b_dequant = dequantize(b_quantized, loaded_quant_info[f'{layer_name}_bias'])
    
    reconstructed_weights.append((w_dequant, b_dequant))
    
    # Calculate reconstruction error
    w_error = np.mean(np.abs(coefs[i] - w_dequant))
    b_error = np.mean(np.abs(intercepts[i] - b_dequant))
    
    print(f"Layer {i}:")
    print(f"  Weight MAE: {w_error:.6f}")
    print(f"  Bias MAE:   {b_error:.6f}")

print("\n✓ Quantization successful!")
print(f"\nCompression Summary:")
print(f"  Original model size: {sum(c.nbytes + ic.nbytes for c, ic in zip(coefs, intercepts)) / 1024:.2f} KB")
print(f"  Quantized model size: {sum(os.path.getsize(output_dir / f) for f in os.listdir(output_dir) if f.endswith('.npy')) / 1024:.2f} KB")
print(f"  Compression ratio: {(sum(c.nbytes + ic.nbytes for c, ic in zip(coefs, intercepts)) / sum(os.path.getsize(output_dir / f) for f in os.listdir(output_dir) if f.endswith('.npy'))):.2f}x")


=== Verification: Dequantization ===

Layer 0:
  Weight MAE: 0.002343
  Bias MAE:   0.000815
Layer 1:
  Weight MAE: 0.002484
  Bias MAE:   0.000805
Layer 2:
  Weight MAE: 0.002164
  Bias MAE:   0.000607

✓ Quantization successful!

Compression Summary:
  Original model size: 1837.08 KB
  Quantized model size: 230.38 KB
  Compression ratio: 7.97x
